# 02 — e-Callisto Dynamic Spectra

Download and plot e-Callisto radio spectra for a solar burst event.

Uses the [`ecallisto-ng`](https://github.com/i4Ds/ecallisto_ng) library.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from ecallisto_ng.data_download.downloader import (
    get_ecallisto_data,
    get_instrument_with_available_data,
)
from ecallisto_ng.plotting.plotting import plot_spectogram_mpl, plot_with_fixed_resolution_mpl

## 1. Define event time window

Set `start` and `end` to bracket the burst. Adjust for your event of interest.

In [ ]:
start = pd.Timestamp('2021-05-07 18:40')
end   = pd.Timestamp('2021-05-07 19:10')

## 2. List available instruments

Shows which e-Callisto stations have data in this window.

In [ ]:
instruments = get_instrument_with_available_data(start, end)
print(f"{len(instruments)} instruments available:")
for name in instruments:
    print(' ', name)

## 3. Download spectra

Returns a dict of `{instrument_name: DataFrame}`.  
Each DataFrame is indexed by time, columns are frequencies in MHz.

In [ ]:
instrument = 'ALASKA-COHOE'   # change to any name from the list above

data = get_ecallisto_data(
    start,
    end,
    instrument_name=instrument,
    verbose=True,
)

print("Downloaded keys:", list(data.keys()))

## 4. Plot dynamic spectrum

In [ ]:
fig = plot_spectogram_mpl(
    data,
    start_datetime=start,
    end_datetime=end,
    fig_size=(12, 5),
    cmap='plasma',
)
plt.tight_layout()
plt.show()

## 5. Compare multiple instruments

Loop over several stations to see how the burst looks from different longitudes.

In [ ]:
stations = instruments[:4]   # first 4 available; adjust as needed

fig, axes = plt.subplots(len(stations), 1, figsize=(12, 4 * len(stations)), sharex=True)

for ax, station in zip(axes, stations):
    d = get_ecallisto_data(start, end, instrument_name=station)
    if not d:
        ax.set_title(f"{station} — no data")
        continue
    df = list(d.values())[0]
    ax.imshow(
        df.T.iloc[::-1],
        aspect='auto',
        extent=[0, df.shape[0], float(df.columns.min()), float(df.columns.max())],
        cmap='plasma',
    )
    ax.set_title(station)
    ax.set_ylabel('Frequency [MHz]')

axes[-1].set_xlabel('Time samples')
fig.suptitle(f"e-Callisto spectra  {start.isoformat()} – {end.isoformat()}")
plt.tight_layout()
plt.show()